In [0]:
spark.sql("use CATALOG `databricks-pyspark` ")

In [0]:
spark.sql("use SCHEMA spark_ns")

In [0]:
spark.sql("DROP TABLE IF EXISTS employee_bronze")
spark.sql("DROP TABLE IF EXISTS employee_silver")
spark.sql("DROP TABLE IF EXISTS cdf_control_table")

In [0]:
%sql
select current_catalog();

In [0]:
%sql
select current_schema()

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS employee_bronze 
          (
              emp_id LONG,
              name STRING,
              salary LONG

          ) using DELTA
           TBLPROPERTIES("delta.enableChangeDataFeed" = "true")
          """)

In [0]:
%sql
describe history employee_source;

In [0]:
spark.sql("""
          insert into employee_bronze values
             (1, 'John', 10000),
             (2, 'Mary', 20000),
             (3, 'Mike', 30000) 
          
          
          """)

In [0]:
spark.sql("describe history employee_bronze").select("version","operation").show()

In [0]:
spark.sql("""
          CREATE TABLE employee_silver using DELTA as SELECT emp_id,name,salary from employee_bronze
          """)

In [0]:
spark.sql("""
          select * from employee_silver
          """).show()

In [0]:
spark.sql("""
          create table if not exists cdf_control_table(tbl_name string,last_processed_version long) using delta
          """)

In [0]:
%sql
insert into cdf_control_table values('employee_bronze',1)

In [0]:
spark.sql("""
INSERT INTO employee_bronze VALUES
(4, 'Sara', 40000)
""")

In [0]:
spark.sql("""
UPDATE employee_bronze
SET salary = 50000
WHERE emp_id = 2
""")

In [0]:
spark.sql("""
DELETE FROM employee_bronze
WHERE emp_id = 3
""")

In [0]:
spark.sql("describe history employee_bronze").show(truncate=False)

In [0]:
last_version = spark.sql("""
                         select max(last_processed_version) from cdf_control_table
                         """).collect()[0][0]
print(last_version)
next_version = last_version + 1


In [0]:
print(next_version)

In [0]:
changes_df = spark.sql(f"""
                       select  emp_id,name,salary,_change_type,_commit_version,_commit_timestamp 
                       from table_changes('employee_bronze',{next_version}) 
                       where _change_type in ('update_postimage','delete','insert') 
                       """) 

In [0]:
changes_df.show()

In [0]:
changes_df.createOrReplaceTempView('employee_changes')

In [0]:
spark.sql("""
          merge into employee_silver as t 
          using employee_changes as s 
          on t.emp_id = s.emp_id
          when MATCHED AND s._change_type = 'update_postimage' then 
          update set t.name = s.name, t.salary= s.salary

          when MATCHED AND s._change_type = 'delete' then
          delete

          WHEN NOT MATCHED AND s._change_type = 'insert' then
          insert (emp_id,name,salary) values (s.emp_id,s.name,s.salary)

          
          """)

In [0]:
spark.sql("""
          select * from employee_silver
          """).show()

In [0]:
max_version_updated = changes_df.selectExpr("max(_commit_version)").collect()[0][0]

In [0]:
spark.sql(f"""
          update cdf_control_table
          set last_processed_version = {max_version_updated}
          where table_name = 'employee_bronze'
          """)


In [0]:
spark.sql(f"""
UPDATE cdf_control_table
SET last_processed_version = {max_version_updated}
WHERE tbl_name = 'employee_bronze'
""")

In [0]:
%sql
select * from cdf_control_table;

In [0]:
spark.sql